# Options Analytics — Backtester Demo

This notebook demonstrates the `Backtester` class from the `options_analytics` library.
Configure the ticker, date range, and strategy parameters in the **Configuration** cell below,
then run all cells to generate the charts and summary table.

**Strategies supported:**
- Long Call
- Long Put
- Covered Call
- Cash-Secured Put
- Wheel

In [33]:
import pandas as pd

from options_analytics.backtester import Backtester
from options_analytics.visualisations import (
    plot_cumulative_pnl,
    plot_daily_pnl,
    plot_portfolio_breakdown,
    plot_portfolio_value
)

## Configuration

All runs share the same ticker and date range. Adjust these to explore different periods.

In [34]:
TICKER       = "AAPL"
START        = "2025-01-01"
END          = "2026-01-01"
INITIAL_CASH = 100_000.0
EXPIRY_DAYS  = 30
STRIKE_PCT   = 1.0   # ATM

## Run all strategies

In [35]:
def run_strategy(strategy: str) -> pd.DataFrame:
    bt = Backtester(
        ticker=TICKER,
        start=START,
        end=END,
        strategy=strategy,
        initial_cash=INITIAL_CASH,
        expiry_days=EXPIRY_DAYS,
        strike_pct=STRIKE_PCT,
    )
    result = bt.run()
    print(f"{'='*55}")
    bt.summary()
    print()
    return result


strategies = ["long_call", "long_put", "covered_call", "cash_secured_put", "wheel"]
results = {s: run_strategy(s) for s in strategies}

Strategy:           long_call
Ticker:             AAPL
Period:             2025-01-01 to 2026-01-01
Cash remaining:     $100,311.84
Shares held:        0
Stock value:        $0.00
Open contracts:     1
Option value:       $5.41
Portfolio value:    $100,317.25
Total return:       0.32%
Annualised return:  0.32%
Sharpe ratio:       -1.19
Max drawdown:       -3.53%
Win rate:           43.60%

Strategy:           long_put
Ticker:             AAPL
Period:             2025-01-01 to 2026-01-01
Cash remaining:     $97,845.51
Shares held:        0
Stock value:        $0.00
Open contracts:     1
Option value:       $680.88
Portfolio value:    $98,526.39
Total return:       -1.47%
Annualised return:  -1.49%
Sharpe ratio:       -1.54
Max drawdown:       -6.55%
Win rate:           32.00%

Strategy:           covered_call
Ticker:             AAPL
Period:             2025-01-01 to 2026-01-01
Cash remaining:     $75,457.97
Shares held:        100
Stock value:        $27,135.58
Open contracts:     1
Op

---
## Chart 1 — Portfolio Value Over Time

Total portfolio value (cash + stock + option mark-to-market) for each strategy.

In [36]:
plot_portfolio_value(results, ticker=TICKER, start=START, end=END).show()

---
## Chart 2 — Cumulative P&L

Gain or loss relative to starting value. Easier to compare strategies directly.

In [37]:
plot_cumulative_pnl(results, ticker=TICKER, start=START, end=END).show()

---
## Chart 3 — Daily P&L

One subplot per strategy. Green bars = profitable days, red bars = losing days.

In [38]:
plot_daily_pnl(results, ticker=TICKER, start=START, end=END).show()

---
## Chart 4 — Portfolio Breakdown

Cash, stock value, and option mark-to-market stacked for a single strategy.
Change `FOCUS_STRATEGY` to inspect a different one.

In [39]:
FOCUS_STRATEGY = "covered_call"
plot_portfolio_breakdown(results[FOCUS_STRATEGY], FOCUS_STRATEGY, ticker=TICKER).show()

---
## Summary Table

Final portfolio value, total return, and max drawdown for each strategy.

In [40]:
rows = []
for strategy, df in results.items():
    if df.empty:
        rows.append({
            "Strategy": strategy.replace("_", " ").title(),
            "Final Value ($)": "BUST",
            "Total Return": "BUST",
            "Max Drawdown": "—",
        })
    else:
        initial  = df["total_value"].iloc[0]
        final    = df["total_value"].iloc[-1]
        ret      = (final - initial) / initial
        rolling_max = df["total_value"].cummax()
        drawdown    = ((df["total_value"] - rolling_max) / rolling_max).min()
        rows.append({
            "Strategy": strategy.replace("_", " ").title(),
            "Final Value ($)": f"{final:,.2f}",
            "Total Return": f"{ret:.2%}",
            "Max Drawdown": f"{drawdown:.2%}",
        })

pd.DataFrame(rows).set_index("Strategy")

,Final Value ($),Total Return,Max Drawdown
Strategy,,,
Long Call,"100,317.25",0.32%,-3.53%
Long Put,"98,526.39",-1.47%,-6.55%
Covered Call,"102,588.14",2.59%,-5.11%
Cash Secured Put,"101,473.61",1.47%,-5.32%
Wheel,"101,643.38",1.64%,-5.22%
